# 组员1：ImageNet图片与双模型预测
**研究问题：** 同一批ImageNet图片上，ResNet50与MobileNetV3 Large的判断依据是否相似？出错时关注区域有什么不同？

本notebook完成共享数据和预测，供后续XAI实验使用。最终20张，四种预测情况各5张。预测不能直接证明模型关注了哪个区域。

## 数据、配置与筛选规则
原图来自MindSpore托管的ImageNet验证集镜像；真实synset来自TensorFlow models验证集标签文件，按原始图片编号查询；类别编号映射来自TensorFlow托管的ImageNet class-index文件。

候选池固定为验证集原始编号1–128。各组按原始编号升序，选择前5个不同真实类别，不参考热力图或confidence高低。A=两者都对，B=仅ResNet50对，C=仅MobileNetV3对，D=两者都错。

ResNet50_Weights.IMAGENET1K_V1与MobileNet_V3_Large_Weights.IMAGENET1K_V1；RGB，短边256、中心裁剪224，官方归一化；eval()。confidence为top-1类别softmax分数；类别ID为0–999。逐张来源与校验值见manifest.json。

**限制：** 这是按预测对错选取的案例集，不能估计总体准确率；原始128张也只是一个小型非随机候选池。对错依据数据集单标签定义，语义相近和多物体案例需要结合图片解释。

来源：[图片](https://tools.mindspore.cn/dataset/workspace/mindspore_dataset/imagenet/imagenet_original/val/) · [标签](https://github.com/tensorflow/models/blob/master/research/slim/datasets/imagenet_2012_validation_synset_labels.txt) · [类别映射](https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json) · [ResNet50](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html) · [MobileNetV3](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_large.html)

In [ ]:
from pathlib import Path
import os, json, hashlib
import pandas as pd
import numpy as np
from IPython.display import display, Image

ROOT = Path.cwd() / 'member1'
assert (ROOT / 'manifest.json').exists(), '请从仓库根目录启动本notebook。'
manifest = json.loads((ROOT / 'manifest.json').read_text(encoding='utf-8'))
saved = pd.read_csv(ROOT / 'predictions.csv')
for row in manifest:
    assert hashlib.sha256((ROOT / row['image_path']).read_bytes()).hexdigest() == row['sha256']
assert saved.groupby('group').size().to_dict() == dict(A=5, B=5, C=5, D=5)
columns = ['image_id', 'group', 'ground_truth_class', 'resnet50_pred_class',
           'resnet50_confidence', 'mobilenetv3_pred_class', 'mobilenetv3_confidence']
display(saved[columns].style.format({'resnet50_confidence': '{:.2%}', 'mobilenetv3_confidence': '{:.2%}'}))

In [ ]:
print('Contact sheet omitted from public package; see manifest.json for sample metadata.')

## 本地复现预测
首次在新电脑或新缓存目录运行将下载两套官方权重。当前电脑已有缓存；环境变量MEMBER1_MODEL_CACHE可指定已有缓存的位置。predict.py保留了模型、预处理和完整推理实现。

此处inference_mode只用于预测；需要梯度的XAI应重新前向计算。

In [ ]:
from member1.predict import predict
cache_dir = Path(os.environ.get('MEMBER1_MODEL_CACHE', str(ROOT / 'model_cache')))
_ = predict(ROOT / 'images', ROOT / 'manifest.json',
            ROOT / 'predictions_rerun.csv', cache_dir)
rerun = pd.read_csv(ROOT / 'predictions_rerun.csv')
assert saved['original_filename'].tolist() == rerun['original_filename'].tolist()
for column in ['resnet50_pred_id', 'mobilenetv3_pred_id', 'group']:
    assert saved[column].tolist() == rerun[column].tolist(), column
for column in ['resnet50_confidence', 'mobilenetv3_confidence']:
    np.testing.assert_allclose(saved[column], rerun[column], rtol=1e-5, atol=1e-6)
print('复现通过：20张图片的预测类别、分组和confidence与交接表一致。')

## 结果说明与交接
128张候选图的实际分组是A=85、B=11、C=9、D=23；最终每组5张。完整候选预测在references/all_128_candidate_predictions.csv。

这说明我们已准备两模型判断一致和不一致、正确和错误的案例。最终样本的组别比例是人为平衡的，因此不报告其总体准确率，也不据此判定哪个模型更好。

**成员2/3/4：** 主解释分别使用resnet50_target_id、mobilenetv3_target_id；B/C/D补充用common_target_id解释同一真实类别。图注写明模型、方法、目标类别及pred/gt。使用相同的224中心裁剪坐标。input_crops仅用于展示，推理从原图做官方预处理。

**成员5：** 可将本notebook的数据、配置、图和结果整合到最终版本。关注区域的结论须在XAI实验后填写。

**组员1仍需亲自完成：** 将交接包分享给队友，制作一页slides，录制约20秒介绍，并准备解释整体研究问题与实验限制。

个别案例解读：D01的真实类别为alp（高山），但原图同时有滑雪者与滑雪板，两模型预测ski；D02的cradle（摇篮）与模型预测crib（婴儿床）语义相近。它们属于“与数据集标签不一致”，分析时应说明多物体或细分类标签的影响，不能直接断言模型完全没看懂图片。